In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 17 consolidation — Fit on anchors, evaluate elsewhere

Read `sessions/17_session.md` first. This is an executed reference, not an assessment of you.

**Evidence:** Real reference point arrangement; target transforms and held-out deformations are generated controls.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT/'data/raw/iris.csv').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Start Jupyter in the extracted listening_to_shape_lab folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))


## Step 1

Generate a known transformation of actual reference point locations. The targets are explicitly synthetic.

Before running: write your expected result and name one way this check could fail.

In [3]:
from shape_lab.stereo import reference_sample, point_cloud
from shape_lab.geometry import exp_so3, fit_similarity
_, _, disparity, calibration = reference_sample()
X, _ = point_cloud(disparity, calibration, stride=70)
X = X[:45]
true_scale = 1.7; true_R = exp_so3([.12,-.08,.05]); true_t = np.array([.2,-.1,.3])
Y = true_scale*X@true_R.T + true_t
anchor = np.arange(len(X)) % 2 == 0
assert anchor.sum()>=3 and (~anchor).sum()>=3

## Step 2

Fit only on anchors, then evaluate the rest.

Before running: write your expected result and name one way this check could fail.

In [4]:
scale, rotation, translation = fit_similarity(X[anchor], Y[anchor])
predicted = scale*X@rotation.T + translation
clean_error = np.linalg.norm(predicted[~anchor]-Y[~anchor], axis=1)
assert np.allclose(scale, true_scale)
assert np.max(clean_error)<1e-10
print('clean held-out maximum:', clean_error.max())

clean held-out maximum: 4.784275514960799e-15


## Step 3

Change only held-out targets. The anchor fit cannot expose this local deformation.

Before running: write your expected result and name one way this check could fail.

In [5]:
warped = Y.copy(); warped[~anchor] += [.03, 0., 0.]
s, r, t = fit_similarity(X[anchor], warped[anchor])
fit_prediction = s*X@r.T + t
fit_error = np.linalg.norm(fit_prediction[anchor]-warped[anchor], axis=1)
holdout_error = np.linalg.norm(fit_prediction[~anchor]-warped[~anchor], axis=1)
assert fit_error.max()<1e-10
assert np.allclose(holdout_error, .03)
result = {'anchors': int(anchor.sum()), 'held_out': int((~anchor).sum()),
          'anchor_max_error': float(fit_error.max()), 'warped_holdout_mean_error': float(holdout_error.mean()),
          'targets_are_generated': True, 'independent_physical_validation': False}

## Keep execution evidence separate from learning evidence

The following saves this reference calculation. Write your own explanation in `my_work/`; no learner score is assigned.

In [6]:
result['stage'] = 17
result['execution_role'] = 'reference_consolidation'
result['data_role'] = 'Real reference point arrangement; target transforms and held-out deformations are generated controls.'
output = ROOT/'reports/consolidation_v4/17.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
print('Saved reference result:', output.relative_to(ROOT))


Saved reference result: reports/consolidation_v4/17.json
